In [2]:
import requests as req
from bs4 import BeautifulSoup as bs
from datetime import datetime
import csv
import os # Added os for file operations
hades = {'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/106.0.0.0 Safari/537.36'}

In [6]:
def scrape_detik(hal, query, start_year, end_year):
    file_path = 'detik2020.csv'
    file_exists = os.path.isfile(file_path)

    # If the file does not exist or is empty, write headers.
    # If it exists and has content, we assume headers are already there
    # or the user intends to append. For a new specific scrape,
    # we should ensure headers are present if the file is new/empty.
    if not file_exists or os.stat(file_path).st_size == 0:
        mode = 'w' # Write mode to create/overwrite and add headers
        write_headers = True
    else:
        mode = 'a' # Append mode
        write_headers = False

    with open(file_path, mode, encoding='utf-8', newline='') as file:
        wr = csv.writer(file, delimiter=',')
        if write_headers:
            wr.writerow(['Headline', 'Date', 'Link', 'Content'])

        from_date_str = f'{start_year}/01/01'
        to_date_str = f'{end_year}/12/31'

        a = 1
        for page in range(1, hal):
            url = f'https://www.detik.com/search/searchnews?query={query}&sortby=time&fromdate={from_date_str}&todate={to_date_str}&page={page}'
            print(f"Scraping URL: {url}")

            try:
                ge = req.get(url, headers=hades, timeout=10).text
                sop = bs(ge, 'lxml')
            except requests.exceptions.RequestException as e:
                print(f"Error fetching search page {url}: {e}")
                continue # Skip to next page

            li = sop.find('div', class_='list media_rows list-berita')
            if not li:
                print(f"No articles list found on page {page} for query '{query}' from {start_year}-{end_year}. Stopping.")
                break

            lin = li.find_all('article')
            if not lin:
                print(f"No individual articles found in the list on page {page}. Stopping.")
                break

            for x in lin:
                link_element = x.find('a')
                if not link_element or not link_element.has_attr('href'):
                    continue
                link = link_element['href']

                date_element = link_element.find('span', class_='date')
                date_text = ''
                if date_element:
                    try:
                        # Clean date text, typically it's 'day-month-year' after splitting by comma
                        parts = date_element.text.replace('WIB', '').replace('detikNews', '').split(',')
                        if len(parts) > 1:
                            date_text = parts[1].strip()
                        else:
                            date_text = parts[0].strip() # Fallback if no comma
                    except IndexError:
                        date_text = date_element.text.replace('WIB', '').replace('detikNews', '').strip()

                headline_element = link_element.find('h2')
                headline = ''
                if headline_element:
                    headline = headline_element.text.strip()

                # Scrape individual article page
                content_ = ''
                try:
                    ge_ = req.get(link, headers=hades, timeout=10).text
                    sop_ = bs(ge_, 'lxml')
                    content_divs = sop_.find_all('div', class_='detail__body-text itp_bodycontent')

                    if content_divs:
                        full_content_paragraphs = []
                        for div in content_divs:
                            paragraphs = div.find_all('p')
                            full_content_paragraphs.extend([p.text for p in paragraphs])
                        content_ = ''.join(full_content_paragraphs).replace('\n', '').replace('ADVERTISEMENT', '').replace('SCROLL TO RESUME CONTENT', '').strip()
                except requests.exceptions.RequestException as e:
                    print(f"Error fetching content for {link}: {e}")
                    content_ = "Error fetching content."
                except Exception as e:
                    print(f"An unexpected error occurred while processing content for {link}: {e}")
                    content_ = "Error processing content."

                print(f'done[{a}] > {headline[0:50]}...')
                wr.writerow([headline, date_text, link, content_])
                a += 1
        print(f"Scraping completed for query '{query}' from {start_year} to {end_year}.")

In [7]:
scrape_detik(hal=3, query='perlindungan data pribadi', start_year=2020, end_year=2025)

Scraping URL: https://www.detik.com/search/searchnews?query=perlindungan data pribadi&sortby=time&fromdate=2020/01/01&todate=2025/12/31&page=1
No articles list found on page 1 for query 'perlindungan data pribadi' from 2020-2025. Stopping.
Scraping completed for query 'perlindungan data pribadi' from 2020 to 2025.


In [5]:
import pandas as pd
df = pd.read_csv('/content/detik2020.csv')
df

FileNotFoundError: [Errno 2] No such file or directory: '/content/2020.csv'

The `soup` object now contains the parsed HTML of Detik.com. What specific data do you want to scrape from this page (e.g., article titles, links, publication dates, etc.)?